# 1. 소개 (Introduction)

이 튜토리얼에서는 전통적인 언어 모델이 자율적인 에이전트로 진화하는 과정을 배운다. LLM이 단순한 텍스트 생성기에서 관찰-사고-행동 루프를 수행하는 지능형 에이전트로 어떻게 변환되는지 이해하게 될 것이다.

## 환경 설정

먼저 필요한 라이브러리를 설치하고 OpenAI API를 설정한다.

In [ ]:
%pip install -q openai

In [1]:
from utils_openai import *

heading("환경 설정 완료")
kv_print({
    "모델": MODEL,
    "메모리 시스템": "MemoryStream",
    "API 함수": "ask(), chat()",
    "도구 함수": "tool_schema(), run_tools()"
})


────────────────────────────────────────
  환경 설정 완료
────────────────────────────────────────
  모델       gpt-4o-mini
  메모리 시스템  MemoryStream
  API 함수   ask(), chat()
  도구 함수    tool_schema(), run_tools()


## 실습 1: 전통적 LLM — 단일 턴 응답

전통적인 LLM의 가장 간단한 사용 방식이다. 사용자 질문을 입력하면, LLM이 즉시 한 번의 응답을 생성한다. 여기에는 메모리, 도구 사용, 계획 수립이 포함되지 않는다.

In [2]:
# 전통적인 LLM 데모: 간단한 질문 → 즉시 응답

step_print(1, "사용자 질문", "파이썬에서 리스트를 어떻게 만드나?")

# ask() 함수: 간단한 프롬프트 입력 → 텍스트 출력
response = ask(
    prompt="파이썬에서 리스트를 어떻게 만드나?",
    system="당신은 파이썬 초급 튜터다. 간결하고 명확하게 설명한다.",
    temperature=0.7,
    max_tokens=300
)

step_print(2, "LLM 응답", response)

heading("특징")
kv_print({
    "상호작용 방식": "한 번의 질문 → 한 번의 응답",
    "메모리": "없음",
    "도구 사용": "없음",
    "학습/적응": "없음",
    "사용 사례": "빠른 답변, 간단한 질의응답"
})

  [Step 1] 사용자 질문
    파이썬에서 리스트를 어떻게 만드나?
  [Step 2] LLM 응답
    파이썬에서 리스트는 대괄호 `[]`를 사용하여 만들 수 있습니다. 리스트는 여러 개의 값을 저장할 수 있는 자료형입니다. 예를 들어:
    
    ```python
    # 빈 리스트 만들기
    my_list = []
    
    # 여러 값을 가진 리스트 만들기
    numbers = [1, 2, 3, 4, 5]
    fruits = ['사과', '바나나', '체리']
    
    # 리스트 안에 다양한 데이터 타입을 포함할 수 있음
    mixed_list = [1, '사과', 3.14, True]
    ```
    
    리스트를 만들 때, 각 요소는 쉼표 `,`로 구분합니다. 필요에 따라 리스트에 요소를 추가하거나 제거할 수도 있습니다.

────────────────────────────────────────
  특징
────────────────────────────────────────
  상호작용 방식  한 번의 질문 → 한 번의 응답
  메모리      없음
  도구 사용    없음
  학습/적응    없음
  사용 사례    빠른 답변, 간단한 질의응답


## 실습 2: 에이전트 LLM — 다단계 추론

에이전트는 Perceive → Retrieve → Plan → Act → Reflect의 6단계를 거친다. 메모리를 활용하여 더 지능적인 응답을 생성할 수 있다.

In [3]:
# 에이전트 LLM 데모: 메모리를 활용한 다단계 추론

# 메모리 스트림 초기화
memory = MemoryStream()

# ==== 단계 1: PERCEIVE (관찰) ====
step_print(1, "PERCEIVE", "사용자의 배경 정보 수집")

observation_1 = "사용자가 처음으로 파이썬을 배우고 있다."
observation_2 = "사용자는 이전에 C++ 경험이 있다."
observation_3 = "사용자가 리스트 자료구조를 배우고 싶어한다."

memory.add(observation_1, importance=0.8, mem_type='observation')
memory.add(observation_2, importance=0.8, mem_type='observation')
memory.add(observation_3, importance=0.9, mem_type='observation')

print(f"  ✓ {observation_1}")
print(f"  ✓ {observation_2}")
print(f"  ✓ {observation_3}")

# ==== 단계 2: RETRIEVE (검색) ====
step_print(2, "RETRIEVE", "관련 메모리 검색")

query = "파이썬 초급자에게 리스트를 가르칠 때 주의할 점"
retrieved = memory.retrieve(query, top_k=2)

print(f"  쿼리: {query}")
for i, mem in enumerate(retrieved, 1):
    print(f"  {i}. {mem['content']} (중요도: {mem['importance']})")

# ==== 단계 3: PLAN (계획) ====
step_print(3, "PLAN", "검색한 메모리를 바탕으로 계획 수립")

context = "\n".join([m['content'] for m in retrieved])
plan_prompt = f"""사용자 배경:
{context}

사용자에게 파이썬 리스트를 가르칠 때의 단계별 계획을 세운다. C++배열과의 비교를 포함한다."""

plan = ask(
    prompt=plan_prompt,
    system="당신은 파이썬 튜터다. 효과적인 교육 계획을 세운다.",
    max_tokens=300
)

print(f"  {plan}")
memory.add(f"[계획] {plan}", importance=0.7, mem_type='plan')

# ==== 단계 4: ACT (행동) ====
step_print(4, "ACT", "계획에 따라 행동 수행")

action_prompt = f"""계획: {plan}

이 계획에 따라 사용자에게 파이썬 리스트의 기초를 설명한다. 코드 예제를 포함한다."""

response = ask(
    prompt=action_prompt,
    system="당신은 파이썬 튜터다. 명확한 코드 예제와 함께 설명한다.",
    max_tokens=400
)

print(f"  {response[:200]}...")
memory.add(f"[행동 결과] {response}", importance=0.8, mem_type='action')

# ==== 단계 5: REFLECT (반성) ====
step_print(5, "REFLECT", "행동 결과를 분석하고 통찰 저장")

reflection_prompt = f"""방금 파이썬 리스트를 설명했다. 설명의 효과성을 판단하고, 다음에 개선할 점을 생각한다.

설명: {response}

이 설명의 강점과 약점을 분석한다."""

reflection = ask(
    prompt=reflection_prompt,
    system="당신은 자신의 교육을 객관적으로 평가한다.",
    max_tokens=300
)

print(f"  {reflection[:200]}...")
memory.add(f"[반성] {reflection}", importance=0.9, mem_type='reflection')

# ==== 최종 요약 ====
heading("에이전트 메모리 상태")
kv_print({
    "총 메모리 수": len(memory.memories),
    "관찰": sum(1 for m in memory.memories if m.get('mem_type') == 'observation'),
    "계획": sum(1 for m in memory.memories if m.get('mem_type') == 'plan'),
    "행동": sum(1 for m in memory.memories if m.get('mem_type') == 'action'),
    "반성": sum(1 for m in memory.memories if m.get('mem_type') == 'reflection')
})

  [Step 1] PERCEIVE
    사용자의 배경 정보 수집
  ✓ 사용자가 처음으로 파이썬을 배우고 있다.
  ✓ 사용자는 이전에 C++ 경험이 있다.
  ✓ 사용자가 리스트 자료구조를 배우고 싶어한다.
  [Step 2] RETRIEVE
    관련 메모리 검색
  쿼리: 파이썬 초급자에게 리스트를 가르칠 때 주의할 점
  1. 사용자가 리스트 자료구조를 배우고 싶어한다. (중요도: 0.9)
  2. 사용자는 이전에 C++ 경험이 있다. (중요도: 0.8)
  [Step 3] PLAN
    검색한 메모리를 바탕으로 계획 수립
  사용자가 리스트 자료구조를 배우기 위한 단계별 교육 계획을 아래와 같이 세울 수 있습니다. C++ 배열과의 비교를 통해 이해도를 높이는 것을 목표로 합니다.

### 1단계: 리스트의 개념 소개
- **목표**: 파이썬 리스트가 무엇인지 이해하기
- **내용**:
  - 리스트의 정의: 여러 개의 값을 저장할 수 있는 데이터 구조
  - 파이썬의 리스트는 동적 배열(dynamic array)로, 크기가 가변적임을 강조
- **비교**: C++의 배열과의 차이점
  - C++ 배열은 고정 크기이며, 리스트는 크기가 동적으로 변할 수 있음
  - C++에서 배열은 같은 타입의 데이터만 담을 수 있지만, 파이썬 리스트는 다양한 데이터 타입을 포함할 수 있음

### 2단계: 리스트 생성 및 초기화
- **목표**: 리스트를 생성하고 초기화하는 방법 배우기
- **내용**:
  - 리스트 생성 방법: `my_list = []` 또는 `my_list = list()`
  - 요소 초기화: `my_list = [1, 2, 3]`, `my_list = ["apple", 2, 3.5]`
- **비교**: C++에서 배열 초기화 예시와 비교
  - C++:
  [Step 4] ACT
    계획에 따라 행동 수행
  안녕하세요! 파이썬 리스트에 대한 기초를 배우기 위한 단계별 교육 계획을 소개하겠습니다. 각 단계에서 설명과 함께 코드 예제

## 실습 3: 전통 LLM vs 에이전트 LLM 비교

같은 질문에 대해 전통 LLM과 에이전트 LLM이 어떻게 다르게 응답하는지 보자.

In [8]:
# 비교: 같은 질문에 대한 두 가지 접근법

question = "파이썬 딕셔너리와 리스트의 차이를 설명해 주세요."

print("="*60)
heading("시나리오: 파이썬 초급자가 자료구조를 구분하지 못함")
print("="*60)

# 방법 1: 전통 LLM
step_print(1, "전통 LLM", "메모리 없이 일반적인 설명 생성")
traditional_response = ask(
    prompt=question,
    system="당신은 파이썬 튜터다.",
    max_tokens=300
)
print(traditional_response)

# 방법 2: 에이전트 LLM (메모리 활용)
step_print(2, "에이전트 LLM", "사용자 배경을 고려한 맞춤형 설명")

# 사용자 배경 정보 추가
agent_memory = MemoryStream()
agent_memory.add("사용자는 파이썬 초급자다.", importance=0.8, mem_type='context')
agent_memory.add("사용자는 C++에서 배열을 배웠다.", importance=0.8, mem_type='context')
agent_memory.add("사용자는 구체적인 예제로 배우기를 좋아한다.", importance=0.9, mem_type='context')

# 메모리를 활용한 프롬프트
context_info = "\n".join([m['content'] for m in agent_memory.retrieve(question, top_k=3)])

agentic_prompt = f"""사용자 정보:
{context_info}

질문: {question}

사용자의 배경을 고려하여 맞춤형으로 설명한다. 특히 C++ 배열과 비교하면서, 구체적인 예제를 보여준다."""

agentic_response = ask(
    prompt=agentic_prompt,
    system="당신은 학생의 배경을 고려하는 파이썬 튜터다.",
    max_tokens=400
)
print(agentic_response)

# 비교 분석
kv_print({
    "전통 LLM": "일반적이고 중립적인 설명",
    "에이전트 LLM": "사용자 배경을 고려한 맞춤형 설명",
    "메모리 활용": "에이전트는 사용자 정보를 기억하고 활용",
    "학습 곡선": "에이전트는 반복될수록 더 나은 응답 제공"
})


────────────────────────────────────────
  시나리오: 파이썬 초급자가 자료구조를 구분하지 못함
────────────────────────────────────────
  [Step 1] 전통 LLM
    메모리 없이 일반적인 설명 생성
파이썬에서 딕셔너리(dictionary)와 리스트(list)는 두 가지 중요한 데이터 구조이며, 각각의 목적과 사용 방식이 다릅니다. 아래에서 이 두 가지의 주요 차이점을 설명하겠습니다.

### 1. 데이터 구조
- **리스트(List)**: 순서가 있는 항목의 집합입니다. 항목은 인덱스를 통해 접근할 수 있으며, 중복된 값을 허용합니다. 리스트는 대괄호 `[]`로 정의합니다.
  ```python
  my_list = [1, 2, 3, 4, 5]
  ```

- **딕셔너리(Dictionary)**: 키-값 쌍의 집합으로, 각 키는 고유해야 하며, 값을 통해 데이터를 저장합니다. 딕셔너리는 중복된 키를 허용하지 않습니다. 중괄호 `{}`로 정의합니다.
  ```python
  my_dict = {'name': 'Alice', 'age': 25, 'city': 'Seoul'}
  ```

### 2. 데이터 접근
- **리스트**: 인덱스를 사용하여 항목에 접근합니다. 인덱스는 0부터 시작합니다.
  ```python
  first_item = my_list[0]  # 1
  ```

- **딕셔너리**: 키를 사용하여 값을 접근합니다.
  ```
  [Step 2] 에이전트 LLM
    사용자 배경을 고려한 맞춤형 설명
안녕하세요! 파이썬의 딕셔너리와 리스트의 차이를 C++의 배열과 비교하여 설명해 드리겠습니다.

### 1. 리스트 (List)

**리스트**는 순서가 있는 데이터의 집합으로, 같은 타입 또는 다른 타입의 요소들을 저장할 수 있습니다. 리스트는 C++의 배열과 비슷하지만, 크기를 동적으로 조절할 수 있다는 점에서 차이가 있습니다.

#### 예제: 리스트 생성 및 사